# container-provision — Colab Translation Worker
Self-contained worker : start Ollama, run bakery Gradle translation task on GPU, push results.
Outputs `TRANSLATION_DONE=ok` on completion.
Caches model + Gradle + deps on Google Drive (persistent across Colab sessions).

## 0. Mount Google Drive (cache)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/.colab-cache'
GRADLE_DIR = f'{CACHE}/gradle-9.6.1'
GRADLE_ZIP = f'{CACHE}/gradle-9.6.1-bin.zip'
GRADLE_USER_HOME = f'{CACHE}/gradle-home'
M2 = f'{CACHE}/.m2/repository'
import os, subprocess, time
os.makedirs(GRADLE_USER_HOME, exist_ok=True)
os.makedirs(M2, exist_ok=True)
os.environ['GRADLE_USER_HOME'] = GRADLE_USER_HOME
subprocess.run(['ln', '-sfn', M2, os.path.expanduser('~/.m2')])
print('Drive mounted, cache dirs ready.')

## 1. Install Ollama directly (no Docker — Colab lacks nested overlayfs)


In [ ]:
# Install zstd + Ollama binary
!apt-get update -qq 2>&1 | tail -1 && apt-get install -y zstd 2>&1 | tail -3
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5
print('Install done.')

## 2. Start Ollama server (GPU + Drive model cache)


In [ ]:
OLLAMA_MODELS = f'{CACHE}/ollama/models'
os.makedirs(OLLAMA_MODELS, exist_ok=True)
# Kill any existing ollama
subprocess.run(['pkill', 'ollama'], capture_output=True)
time.sleep(2)
# Start ollama serve (port natif 11434 — Colab isolé, pas de contrainte de plage)
!OLLAMA_MODELS={OLLAMA_MODELS} nohup ollama serve > /tmp/ollama.log 2>&1 &
print('Ollama server starting on 11434...')

## 3. Pull translation model + wait for ready


In [ ]:
import urllib.request
# Pull model
!ollama pull translategemma:4b 2>&1
print('Model pull done (or from cache).')
# Wait for Ollama ready
for i in range(150):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags')
        print('Ollama ready.')
        break
    except Exception as e:
        if i % 10 == 0:
            print(f'  Waiting... ({i*2}s)')
        time.sleep(2)
else:
    print('Ollama not ready after 300s')
    with open('/tmp/ollama.log') as f:
        print(f.read()[-500:])

## 5. Install JDK

In [ ]:
# Install Temurin JDK 25
!wget -qO /tmp/jdk25.tar.gz 'https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse'
!mkdir -p /opt/jdk && tar xzf /tmp/jdk25.tar.gz -C /opt/jdk --strip-components=1
os.environ['JAVA_HOME'] = '/opt/jdk'
os.environ['PATH'] = '/opt/jdk/bin:' + os.environ['PATH']
!java --version
print('JDK 25 Temurin ready.')

## 6. Create Gradle project with bakery plugin

In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('github-api-key-cheroliv')
print(f"Token loaded: {'yes' if GITHUB_TOKEN else 'no'}")
WORK = '/content/colab-translate'
os.makedirs(WORK, exist_ok=True)

In [ ]:
%%writefile {WORK}/settings.gradle.kts
pluginManagement {
    repositories { mavenCentral(); gradlePluginPortal() }
}
dependencyResolutionManagement {
    repositories { mavenCentral() }
}
rootProject.name = "colab-translate"

In [ ]:
%%writefile {WORK}/build.gradle.kts
buildscript {
    repositories {
        mavenCentral()
        gradlePluginPortal()
    }
    dependencies {
        classpath(platform("education.cccp:workspace-bom:0.0.6"))
        classpath("education.cccp:bakery-plugin")
    }
}
apply(plugin = "education.cccp.bakery")

val langs = (findProperty("targetLangs") as? String)?.split(",") ?: listOf("en")
val ollamaUrl = findProperty("iaBaseUrl") as? String ?: "http://localhost:11434"
val siteDir = findProperty("siteDir") as? String ?: error("-PsiteDir=<path> required")

extensions.configure<bakery.BakeryExtension>("bakery") {
    configPath = file(siteDir).resolve("site.yml").absolutePath
    ia { baseUrl = ollamaUrl; modelName = "translategemma:4b"; enabled = true }
    contentI18nMigration {
        sourceDir = "$siteDir/jbake/content"
        outputDir = "$siteDir/content-i18n"
        sourceLanguage = "fr"
        targetLanguages = langs
        dryRun = false
    }
}

## 7. Gradle 9.6.1 (cached on Drive)

In [ ]:
%cd {CACHE}
if not os.path.exists(GRADLE_DIR):
    if not os.path.exists(GRADLE_ZIP):
        !curl -sLO https://services.gradle.org/distributions/gradle-9.6.1-bin.zip
    !unzip -q gradle-9.6.1-bin.zip
    os.remove(GRADLE_ZIP)
    print('Gradle 9.6.1 downloaded and extracted to Drive cache.')
else:
    print('Gradle 9.6.1 already in Drive cache.')
# Drive is noexec — copy to local filesystem for execution
LOCAL_GRADLE = '/content/gradle-local'
if not os.path.exists(LOCAL_GRADLE):
    import shutil
    shutil.copytree(GRADLE_DIR, LOCAL_GRADLE)
    print('Copied Gradle to local filesystem (executable).')
os.chmod(os.path.join(LOCAL_GRADLE, 'bin', 'gradle'), 0o755)
GRADLE_BIN = os.path.join(LOCAL_GRADLE, 'bin', 'gradle')

## 8. Run translation

In [ ]:
# Clone site
LANG = 'en'
REPO_URL = userdata.get('REPO_OFFICE') or 'https://github.com/cheroliv/office.git'
SITE_NAME = userdata.get('site_name') or 'cheroliv.com'
SITE_ROOT = '/content/office'
SITE_DIR = f'{SITE_ROOT}/sites/{SITE_NAME}'
if GITHUB_TOKEN:
    import urllib.parse
    parsed = urllib.parse.urlparse(REPO_URL)
    AUTH_REPO = f'{parsed.scheme}://cheroliv:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}'
else:
    AUTH_REPO = REPO_URL
!rm -rf {SITE_ROOT}
!git clone --depth=1 {AUTH_REPO} {SITE_ROOT} 2>&1
import os.path
if not os.path.isdir(SITE_DIR):
    raise RuntimeError(f'Failed to clone — {SITE_DIR} not found')
print(f'Site cloned to {SITE_DIR}')

In [ ]:
if not os.path.isdir(SITE_DIR):
    raise RuntimeError(f'Site not cloned to {SITE_DIR} — aborting translation')
# Show site structure
!ls {SITE_DIR}/jbake/content/ | head -10
%cd {WORK}
r = subprocess.run([GRADLE_BIN, 'migrateContentI18n',
    f'-PsiteDir={SITE_DIR}',
    f'-PtargetLangs={LANG}',
    '-PiaBaseUrl=http://localhost:11434',
    '--stacktrace'], capture_output=True, text=True)
print('STDOUT:', r.stdout[-2000:] if r.stdout else '(empty)')
print('STDERR (first 2k):', r.stderr[:2000] if r.stderr else '(empty)')
print('STDERR (last 2k):', r.stderr[-2000:] if r.stderr else '(empty)')
if r.returncode != 0:
    raise RuntimeError(f'Translation failed (exit {r.returncode})')
print(f'Translation done. Output: {SITE_DIR}/content-i18n')
!ls {SITE_DIR}/content-i18n/ 2>/dev/null | head -10 || echo '(no content-i18n dir)'

## 9. Push results

In [ ]:
if not os.path.isdir(SITE_ROOT + '/.git'):
    raise RuntimeError(f'{SITE_ROOT} is not a git repo — cannot push')
%cd {SITE_ROOT}
!git config user.email 'cheroliv@users.noreply.github.com'
!git config user.name 'cheroliv'
!git add sites/{SITE_NAME}/content-i18n && git commit -m "i18n({LANG}): {SITE_NAME} translation via Colab + translategemma:4b" || echo 'Nothing to commit'
if GITHUB_TOKEN:
    import urllib.parse
    parsed = urllib.parse.urlparse(REPO_URL)
    AUTH_URL = f'{parsed.scheme}://cheroliv:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}'
    !git remote set-url origin {AUTH_URL}
    !git push origin main 2>&1
    print('Push done.')
else:
    print('Push skipped (no token)')

## Done

In [ ]:
print('TRANSLATION_DONE=ok')